In [ ]:
# 🧩 Cell 1 — Install & imports
!pip install -q datasets huggingface_hub

from datasets import load_dataset
from huggingface_hub import login

# 🧩 Cell 2 — Log in to Hugging Face (run once and paste your token)
login()

# ─────────────────────────────────────────────
# Helpers to clean hemistichs and build verses
# ─────────────────────────────────────────────

def clean_hemistich(h):
    if h is None:
        return ""
    return h.replace("<s>", "").replace("<a>", "").strip()

def hemistich_list_to_verses(lst):
    """
    Takes a flat list:
      [s1, a1, s2, a2, ...]
    and returns:
      ["s1   a1", "s2   a2", ...]
    """
    if lst is None:
        return []
    verses = []
    n = len(lst)
    for i in range(0, n, 2):
        pair = lst[i:i+2]
        if len(pair) < 2:
            # ignore dangling hemistich if odd length
            continue
        h1 = clean_hemistich(pair[0])
        h2 = clean_hemistich(pair[1])
        verses.append(f"{h1}   {h2}")
    return verses

def add_clean_verses(example):
    # target_verse: list of 2 hemistichs for the *current* verse
    tv_list = example.get("target_verse", None)
    tv_verses = hemistich_list_to_verses(tv_list)
    target_verse_clean = tv_verses[0] if len(tv_verses) > 0 else ""

    # previous_verses: flat list of hemistichs for all previous verses
    pv_list = example.get("previous_verses", None)
    previous_verses_clean = hemistich_list_to_verses(pv_list)

    return {
        "target_verse_clean": target_verse_clean,
        "previous_verses_clean": previous_verses_clean,
    }

# 🧩 Cell 3 — Process and push all three datasets

# Original repos (with underscore)
source_ids = {
    "train": "Shaer-AI/ashaar-training_split",
    "test": "Shaer-AI/ashaar-testing_split",
    "validation": "Shaer-AI/ashaar-validation_split",
}

# New cleaned repos (with hyphen)
target_ids = {
    "train": "Shaer-AI/ashaar-training-split",
    "test": "Shaer-AI/ashaar-testing-split",
    "validation": "Shaer-AI/ashaar-validation-split",
}

cleaned_datasets = {}

for split_name in ["train", "test", "validation"]:
    src_id = source_ids[split_name]
    tgt_id = target_ids[split_name]

    print(f"\n🔹 Loading {src_id} ...")
    ds = load_dataset(src_id, split="train")  # each of these repos has a single 'train' split

    print("Before:", ds.column_names)
    ds = ds.map(add_clean_verses)
    print("After: ", ds.column_names)

    # Quick sanity check
    print("Example row:")
    print("- raw target_verse:", ds[0]["target_verse"])
    print("- clean target_verse:", ds[0]["target_verse_clean"])
    print("- clean previous_verses:", ds[0]["previous_verses_clean"][:3], "...")

    cleaned_datasets[split_name] = ds

    # 👉 If you want to test first without pushing, comment the next 2 lines
    print(f"⬆️ Pushing cleaned dataset to {tgt_id} ...")
    ds.push_to_hub(tgt_id)

print("\n✅ Done: cleaned datasets pushed as:")
for split_name, repo_id in target_ids.items():
    print(f"- {split_name}: {repo_id}")



🔹 Loading Shaer-AI/ashaar-training_split ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Before: ['poem_title', 'poem_meter', 'poem_theme', 'poem_url', 'poet_name', 'poet_description', 'poet_url', 'poet_era', 'poet_location', 'poem_language_type', 'num_verses', 'poem_id', 'poem_description', 'target_verse', 'previous_verses', 'sequence_number']
After:  ['poem_title', 'poem_meter', 'poem_theme', 'poem_url', 'poet_name', 'poet_description', 'poet_url', 'poet_era', 'poet_location', 'poem_language_type', 'num_verses', 'poem_id', 'poem_description', 'target_verse', 'previous_verses', 'sequence_number', 'target_verse_clean', 'previous_verses_clean']
Example row:
- raw target_verse: ['ما على القَلْبِ بعْدكُمْ منْ جُناحِ<s>', 'أنْ يُرى طائِراً بغيْرِ جَناحِ<a>']
- clean target_verse: ما على القَلْبِ بعْدكُمْ منْ جُناحِ   أنْ يُرى طائِراً بغيْرِ جَناحِ
- clean previous_verses: [] ...
⬆️ Pushing cleaned dataset to Shaer-AI/ashaar-training-split ...


Uploading the dataset shards:   0%|          | 0/13 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/85 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/70 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :  52%|#####1    | 13.3MB / 25.7MB            


✅ Done: cleaned datasets pushed as:
- train: Shaer-AI/ashaar-training-split
- test: Shaer-AI/ashaar-testing-split
- validation: Shaer-AI/ashaar-validation-split
